In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, r2_score, confusion_matrix
import numpy as np
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
df = pd.read_excel('/content/All_data_final.xlsx')

# drop all rows with nan
df=df.dropna(axis=0)

In [ ]:
selected_columns=['accident type', '发生地种类', 'season', 'wind speed level', 'dayNight',
       'fatality', '作业情况', '损伤种类', '损伤位置', 'wind direction', '能见度等级', '是否超速',
       '路况种类', 'ship type', '船体材料', '主机类型', '船长', '船宽', '设备故障', '环境恶劣']

In [ ]:
def filter_rare_values(df, selected_cols, threshold=5):
    for col in selected_columns:
        value_counts = df[col].value_counts()
        df = df[df[col].map(value_counts) >= threshold]  # Keep rows where the value appears ≥ threshold
    return df
df=filter_rare_values(df,selected_columns)
df.drop(df[df['accident type'] == 'Others'].index, inplace=True)

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertModel.from_pretrained("bert-base-chinese")

def get_embedding(text):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean pooling of the last hidden state for sentence-level embeddings
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_embedding(x))
Traffic_Embeddings=df['事发水域路况'].fillna("").apply(lambda x: get_embedding(x))

df['Bert_cause'],df['Bert_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000)

def get_tfidf_embedding(text):
    # Vectorize the text using the fitted TF-IDF vectorizer
    embedding = vectorizer.transform([text]).toarray().squeeze()
    return embedding

vectorizer.fit(df['Cause of Accident'].fillna(""))
# Apply the TF-IDF embedding function to each row in the DataFrame column
Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_tfidf_embedding(x))


vectorizer.fit(df['事发水域路况'].fillna(""))
Traffic_Embeddings= df['事发水域路况'].fillna("").apply(lambda x: get_tfidf_embedding(x))

df['Tfidf_cause'],df['Tfidf_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
df = df.drop(['Cause of Accident', '事发水域路况'], axis=1)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

class TextEmbeddingClusterModel:
    def __init__(self, n_clusters=7, seed=1):
        self.n_clusters = n_clusters
        self.seed = seed

        self.kmeans_cause_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_cause_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)

    def train(self, df):
        # Expecting these columns to contain array-like vectors
        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            self.kmeans_cause_bert.fit(cause_bert_matrix)
        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            self.kmeans_traffic_bert.fit(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            self.kmeans_cause_tfidf.fit(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            self.kmeans_traffic_tfidf.fit(traffic_tfidf_matrix)


        return self

    def predict(self, df):
        cause_bert_clusters = None
        traffic_bert_clusters = None
        cause_tfidf_clusters = None
        traffic_tfidf_clusters = None

        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            cause_bert_clusters = self.kmeans_cause_bert.predict(cause_bert_matrix)

        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            traffic_bert_clusters = self.kmeans_traffic_bert.predict(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            cause_tfidf_clusters = self.kmeans_cause_tfidf.predict(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            traffic_tfidf_clusters = self.kmeans_traffic_tfidf.predict(traffic_tfidf_matrix)

        return cause_tfidf_clusters, traffic_tfidf_clusters, cause_bert_clusters, traffic_bert_clusters



In [ ]:
def cross_validate(seed, X_train, y_train):
    models = {
    "GBM": (
        GradientBoostingClassifier(),
        {
            'n_estimators': [100, 150, 200],
            'learning_rate': [0.03, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'min_samples_split': [2, 4, 6],
            'min_samples_leaf': [1, 2],
            'subsample': [0.8, 1.0],
            'max_features': ['sqrt', 'log2']
        }
    ),
    "RF": (
        RandomForestClassifier(),
        {
            'n_estimators': [100, 150, 200],
            'max_depth': [None, 15, 25],
            'min_samples_split': [2, 4, 6],
            'min_samples_leaf': [1, 2],
            'bootstrap': [True],
            'criterion': ['gini'],
            'max_features': ['sqrt', 'log2'],
            'max_leaf_nodes': [None, 20, 30]
        }
    )
}


    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)

    # Initialize tracking
    best_model_name = None
    best_avg_score = -np.inf
    best_params = None
    best_model_class = None

    for name, (model, param_grid) in models.items():
        print(f"Training model: {name}")
        fold_accuracies = []
        best_param_candidates = []

        for train_idx, test_idx in skf.split(X_train, y_train):
            fold_trainx, fold_trainy = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx].copy()
            fold_testx, fold_testy = X_train.iloc[test_idx].copy(), y_train.iloc[test_idx].copy()

            # Retrain embedding model on current fold's training data
            embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
            embedding_model.train(fold_trainx)

            try:
                tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(fold_trainx)
                if tfidf_cause is not None:
                    fold_trainx['Tfidf_cause'] = tfidf_cause
                if tfidf_traffic is not None:
                    fold_trainx['Tfidf_traffic'] = tfidf_traffic
                if bert_cause is not None:
                    fold_trainx['Bert_cause'] = bert_cause
                if bert_traffic is not None:
                    fold_trainx['Bert_traffic'] = bert_traffic
            except Exception as e:
                print(f"Warning during train embedding on fold: {e}")

            try:
                tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(fold_testx)
                if tfidf_cause is not None:
                    fold_testx['Tfidf_cause'] = tfidf_cause
                if tfidf_traffic is not None:
                    fold_testx['Tfidf_traffic'] = tfidf_traffic
                if bert_cause is not None:
                    fold_testx['Bert_cause'] = bert_cause
                if bert_traffic is not None:
                    fold_testx['Bert_traffic'] = bert_traffic
            except Exception as e:
                print(f"Warning during test embedding on fold: {e}")

            try:
                grid_search = GridSearchCV(model, param_grid=param_grid, cv=3)
                grid_search.fit(fold_trainx, fold_trainy)

                preds = grid_search.best_estimator_.predict(fold_testx)
                acc = accuracy_score(fold_testy, preds)
                fold_accuracies.append(acc)

                best_param_candidates.append(grid_search.best_params_)
            except Exception as e:
                print(f"Grid search or prediction failed: {e}")

        avg_score = np.mean(fold_accuracies)
        print(f"{name} average training CV accuracy: {avg_score:.4f}")

        if avg_score > best_avg_score:
            best_avg_score = avg_score
            best_model_name = name
            best_model_class = model.__class__

            # Take the most common best params from 4 folds
            best_index = np.argmax(fold_accuracies)
            best_params = best_param_candidates[best_index]


    # Refit the model on the **full training data** using best model type and parameters
    print(f"Refitting best model '{best_model_name}' with best parameters on full training data")
    final_model = best_model_class(**best_params)

    # Re-embed the full training set
    embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
    embedding_model.train(X_train)

    try:
        tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(X_train)
        if tfidf_cause is not None:
            X_train['Tfidf_cause'] = tfidf_cause
        if tfidf_traffic is not None:
            X_train['Tfidf_traffic'] = tfidf_traffic
        if bert_cause is not None:
            X_train['Bert_cause'] = bert_cause
        if bert_traffic is not None:
            X_train['Bert_traffic'] = bert_traffic

    except Exception as e:
        print(f"Embedding failed on training data: {e}")


    # Fit the final model
    final_model.fit(X_train, y_train)

    return final_model, best_avg_score




In [ ]:
def type_prediction(df, seed_list, output_csv="409relevel_accident_with_concated_full_data"):
    results = []

    for seed in seed_list:
        print(f"Started seed {seed}")

        # 1. Split the data
        df_train, df_test = train_test_split(df, test_size=0.3, random_state=seed, stratify=df['accident level']) # accident type


        # 2. Separate y and X
        y1 = df_train['accident type']
        X1 = df_train.drop(columns=['fatality', 'accident type', 'accident level', '损伤种类', '损伤位置','环境恶劣', '设备故障'])
        y2 = df_test['accident type']
        X2 = df_test.drop(columns=['fatality', 'accident type', 'accident level', '损伤种类', '损伤位置', '环境恶劣', '设备故障'])

        # 3. One-hot encode consistent categorical features
        type_categorical_cols = X1.select_dtypes(include=['object']).columns
        exclude_cols = ['Bert_cause', 'Bert_traffic', 'Tfidf_cause', 'Tfidf_traffic']
        type_categorical_cols = [col for col in type_categorical_cols if col not in exclude_cols]

        X1 = pd.get_dummies(X1, columns=type_categorical_cols, drop_first=True)
        type_expected_columns = X1.columns

        X2 = pd.get_dummies(X2, columns=type_categorical_cols, drop_first=True)
        X2 = X2.reindex(columns=type_expected_columns, fill_value=0)

        # 4. Train models using internal CV
        best_model, best_score = cross_validate(seed, X1, y1)  # no conf matrix needed here

        # 5. Create embedding features for X2 (validation)
        embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
        embedding_model.train(type_train)  # retrain on full training data

        tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(type_val)
        X2['Tfidf_cause'], X2['Tfidf_traffic'] = tfidf_cause, tfidf_traffic
        X2['Bert_cause'], X2['Bert_traffic'] = bert_cause, bert_traffic

        # 6. Predict on validation set and evaluate
        y2_pred = best_model.predict(X2)
        val_acc = accuracy_score(y2, y2_pred)
        conf_matrix = confusion_matrix(y2, y2_pred)

        y3=df_train['accident level']
        X3=df_train.drop(columns=['fatality', 'accident type','accident level', '损伤种类', '损伤位置','环境恶劣', '设备故障'])

        categorical_cols = X3.select_dtypes(include=['object']).columns
        exclude_cols = ['Bert_cause', 'Bert_traffic', 'Tfidf_cause', 'Tfidf_traffic']
        categorical_cols = [col for col in categorical_cols if col not in exclude_cols]

        X3=pd.get_dummies(X3, columns=categorical_cols, drop_first=True)
        expected_columns = X3.columns


        X3_type=best_model.predict_proba(X3)
        prob_cols = [f'prob_class_{i}' for i in range(X3_type.shape[1])]
        X3probs_df = pd.DataFrame(prob_cols, columns=prob_cols, index=X3.index)

        NewX3 = pd.concat([X3, X3probs_df], axis=1)

        y4=df_test['accident level']
        X4=df_test.drop(columns=['fatality', 'accident type','accident level', '损伤种类', '损伤位置','环境恶劣', '设备故障'])

        X4=pd.get_dummies(X4, columns=categorical_cols, drop_first=True)
        X4=X4.reindex(columns=expected_columns, fill_value=0)

        X4['Tfidf_cause'], X4['Tfidf_traffic'] = tfidf_cause, tfidf_traffic
        X4['Bert_cause'], X4['Bert_traffic'] = bert_cause, bert_traffic

        X4_type=best_model.predict_proba(X4)
        X4prob_cols = [f'prob_class_{i}' for i in range(X4_type.shape[1])]
        X4probs_df = pd.DataFrame(X4prob_cols, columns=prob_cols, index=X4.index)

        NewX4 = pd.concat([X4, X4probs_df], axis=1)

        best_accident_model,best_accident_score=cross_validate(seed,NewX3,y3)

        y4_pred=best_accident_model.predict(NewX4)
        acc=accuracy_score(y4,y4_pred)
        matrix=confusion_matrix(y4,y4_pred)

        precision_macro = precision_score(y4, y4_pred, average='macro')
        recall_macro = recall_score(y4, y4_pred, average='macro')

        # 7. Store the result
        # 7. Store the result
        results.append({
            'seed': seed,
            'type_train': best_score
            'type_test': val_acc,
            'type_matrix': np.array2string(conf_matrix, separator=', ')
            'accident_train': best_accident_score
            'accident_test': acc,
            'accident_matrix': np.array2string(matrix, separator=', ')
            'macro_precision': precision_macro,
            'macro_recall': recall_macro,
        })

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    return results_df

seed_pool = [1, 12, 42, 55, 123, 777]
results_df = type_prediction(df, seed_pool)

Started seed 12
['发生地种类', 'season', 'dayNight', '作业情况', 'wind direction', '能见度等级', '是否超速', '路况种类', 'ship type', '船体材料', '主机类型', '船长', '船宽']
Training model: GBM
